














 DERMA-AI v2: DoRT-Net (Domain-Robust Triage Network)

One model, one forward pass, built for skin-lesion **triage** (low / moderate / high) that holds up on **smartphone photos**, not just dermoscopic images.

**How to use this notebook**
1. **First time only:** have your `kaggle.json` API token ready (Kaggle > Settings > API > Create New Token). Section 2 downloads both datasets from Kaggle automatically, so nothing needs to be uploaded to Drive.
2. Run every cell top to bottom, in order.
3. Run the **smoke test** (Section 8) first. It takes a few minutes and confirms everything works before the long runs.
4. Each experiment saves its checkpoint and results to Google Drive (`MyDrive/DERMA-AI-v2/`). If Colab disconnects, reconnect, re-run Sections 1–7, and finished experiments are skipped automatically.
5. Section 10 builds the paper's results table from whatever experiments have finished.

**What DoRT-Net adds on top of a plain EfficientNet-B0**

| Component | Purpose |
|---|---|
| FiLM metadata fusion | Age, sex, and lesion site modulate the image features |
| Prototype attention | Each image is compared to learned "typical case" vectors (no stored graph) |
| Under-triage cost loss | Calling a melanoma "low" is penalised far more than over-caution |
| Smartphone-simulation consistency | The model must agree with itself on a phone-degraded copy of each image |

The **baseline** is the same network with all four switched off, so every comparison is like-for-like.

> Research/educational prototype. Not a medical device and not a diagnosis.

## 1. Setup

In [ ]:
import os, io, glob, json, time, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, recall_score, classification_report, confusion_matrix

SEED = 42
def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"PyTorch {torch.__version__} | device: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if USE_AMP else " -- switch Runtime > Change runtime type > T4 GPU"))

PyTorch 2.11.0+cu130 | device: cuda (Tesla T4)


In [ ]:
# Google Drive: every checkpoint and result is saved here so a disconnect never loses finished work
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/DERMA-AI-v2"
except Exception as e:
    print("Drive not available, saving locally instead:", e)
    WORK_DIR = "/content/derma_v2_out"

CKPT_DIR    = f"{WORK_DIR}/checkpoints"
RESULTS_DIR = f"{WORK_DIR}/results"
DATA_DIR    = "/content/data"
for d in (CKPT_DIR, RESULTS_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)
print("Outputs will be saved to:", WORK_DIR)

Drive not available, saving locally instead: Error: credential propagation was unsuccessful
Outputs will be saved to: /content/derma_v2_out


## 2. Download both datasets from Kaggle

Both datasets download straight onto the Colab disk, so nothing needs to be uploaded:

| Dataset | Kaggle slug | Used for |
|---|---|---|
| HAM10000 | `kmader/skin-cancer-mnist-ham10000` | training, validation, in-domain test |
| PAD-UFES-20 | `mahdavi1202/skin-cancer` | external smartphone test only |

**First session:** the next cell asks you to upload `kaggle.json`, then keeps a copy in your Drive folder (`MyDrive/DERMA-AI-v2/kaggle.json`) so later sessions skip the upload. That file is your private Kaggle key, so don't share that Drive folder.

The Colab disk is wiped when a session ends, so each new session downloads the data again (a few minutes). Checkpoints and results stay safe on Drive.

In [ ]:
import zipfile, subprocess, sys

# ---------------- Kaggle credentials ----------------
KAGGLE_DIR = os.path.expanduser("~/.kaggle")
KAGGLE_JSON = f"{KAGGLE_DIR}/kaggle.json"
KAGGLE_JSON_DRIVE = f"{WORK_DIR}/kaggle.json"
os.makedirs(KAGGLE_DIR, exist_ok=True)

if not os.path.exists(KAGGLE_JSON):
    if os.path.exists(KAGGLE_JSON_DRIVE):
        shutil.copy(KAGGLE_JSON_DRIVE, KAGGLE_JSON)
        print("Using kaggle.json saved in Drive.")
    else:
        from google.colab import files
        print("Upload kaggle.json (Kaggle > Settings > API > Create New Token):")
        uploaded = files.upload()
        token = next((data for name, data in uploaded.items() if name.lower().endswith(".json")), None)
        assert token, "No .json file uploaded. Re-run this cell and pick kaggle.json."
        with open(KAGGLE_JSON, "wb") as f:
            f.write(token)
        shutil.copy(KAGGLE_JSON, KAGGLE_JSON_DRIVE)
        print("Saved a copy to Drive for next session.")
os.chmod(KAGGLE_JSON, 0o600)

if shutil.which("kaggle") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)

# ---------------- download ----------------
HAM_DIR = f"{DATA_DIR}/ham"
PAD_DIR = f"{DATA_DIR}/pad"

def extract_nested_zips(root):
    done = set()
    while True:
        nested = [z for z in glob.glob(f"{root}/**/*.zip", recursive=True) if z not in done]
        if not nested:
            return
        for zpath in nested:
            print("  extracting nested", os.path.basename(zpath))
            with zipfile.ZipFile(zpath) as z:
                z.extractall(os.path.dirname(zpath))
            done.add(zpath)

def kaggle_download(slug, dest, csv_name):
    hits = glob.glob(f"{dest}/**/{csv_name}", recursive=True)
    if hits:
        print(f"{slug}: already on disk")
        return hits[0]
    os.makedirs(dest, exist_ok=True)
    print(f"Downloading {slug} ... (a few minutes)")
    r = subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", dest, "--unzip"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Kaggle download failed for {slug}:\n{r.stderr or r.stdout}\n"
                           "Check kaggle.json is valid, and open the dataset page on kaggle.com once "
                           "while logged in if it asks you to accept terms.")
    extract_nested_zips(dest)
    hits = glob.glob(f"{dest}/**/{csv_name}", recursive=True)
    assert hits, f"{csv_name} not found after downloading {slug}"
    return hits[0]

HAM_CSV = kaggle_download("kmader/skin-cancer-mnist-ham10000", HAM_DIR, "HAM10000_metadata.csv")
PAD_CSV = kaggle_download("mahdavi1202/skin-cancer", PAD_DIR, "metadata.csv")

ham_check = pd.read_csv(HAM_CSV)
pad_check = pd.read_csv(PAD_CSV)
assert {"image_id", "dx", "lesion_id"} <= set(ham_check.columns), f"Unexpected HAM columns: {list(ham_check.columns)}"
assert "diagnostic" in [c.lower() for c in pad_check.columns], f"Unexpected PAD columns: {list(pad_check.columns)}"
print(f"HAM10000 CSV   : {HAM_CSV}  ({len(ham_check)} rows)")
print(f"PAD-UFES-20 CSV: {PAD_CSV}  ({len(pad_check)} rows)")

Upload kaggle.json (Kaggle > Settings > API > Create New Token):


In [ ]:
# ---------------- match every CSV row to its image file ----------------
def index_images(root, exts):
    found = {}
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn.lower().endswith(exts):
                found.setdefault(os.path.splitext(fn)[0], os.path.join(dirpath, fn))
    return found

def image_index(root, exts, csv_ids, name):
    idx = index_images(root, exts)
    matched = sum(i in idx for i in csv_ids)
    print(f"{name}: {matched} / {len(csv_ids)} images matched to the CSV")
    if matched < 0.9 * len(csv_ids):
        raise FileNotFoundError(f"Only {matched} of {len(csv_ids)} {name} images found under {root}. "
                                f"Delete {root} and re-run the previous cell to download again.")
    return idx

ham_ids = ham_check["image_id"].astype(str).tolist()
pad_id_col = next((c for c in pad_check.columns if c.lower() in ("img_id", "image_id", "img", "image")), None)
assert pad_id_col, f"No image-id column in {PAD_CSV}. Columns: {list(pad_check.columns)}"
pad_ids = [os.path.splitext(s)[0] for s in pad_check[pad_id_col].astype(str)]

HAM_IMAGES = image_index(HAM_DIR, (".jpg", ".jpeg"), ham_ids, "HAM10000")
PAD_IMAGES = image_index(PAD_DIR, (".png", ".jpg", ".jpeg"), pad_ids, "PAD-UFES-20")
print("\nBoth datasets ready.")

## 3. HAM10000: tiers and a lesion-level split

HAM10000 contains several photos of the same lesion. The split below keeps **every photo of a lesion in one split only**, so the test set never contains a lesion the model saw in training. This is stricter than a random image split, so scores may be a little lower than earlier notebook runs, but they are honest.

In [ ]:
CLASS_NAMES   = ["nv", "bkl", "df", "vasc", "akiec", "bcc", "mel"]
CLASS_TO_IDX  = {c: i for i, c in enumerate(CLASS_NAMES)}
CLASS_TO_TIER = {"nv": "low", "bkl": "low", "df": "low",
                 "vasc": "moderate", "akiec": "moderate", "bcc": "moderate",
                 "mel": "high"}
TIER_NAMES  = ["low", "moderate", "high"]
TIER_TO_INT = {t: i for i, t in enumerate(TIER_NAMES)}

ham = pd.read_csv(HAM_CSV)
ham["src_path"] = ham["image_id"].astype(str).map(HAM_IMAGES)
ham = ham.dropna(subset=["src_path"]).reset_index(drop=True)
ham["tier_int"] = ham["dx"].map(CLASS_TO_TIER).map(TIER_TO_INT)

def grouped_holdout(df, n_splits, seed=SEED):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    keep_idx, hold_idx = next(sgkf.split(df, df["dx"], groups=df["lesion_id"]))
    return df.iloc[keep_idx].reset_index(drop=True), df.iloc[hold_idx].reset_index(drop=True)

rest_df, test_df  = grouped_holdout(ham, 7)       # ~14% of lesions -> test
train_df, val_df  = grouped_holdout(rest_df, 6)   # ~14% of total -> val

assert not set(train_df.lesion_id) & set(test_df.lesion_id)
assert not set(train_df.lesion_id) & set(val_df.lesion_id)
assert not set(val_df.lesion_id) & set(test_df.lesion_id)

print(f"Images -> train {len(train_df)} | val {len(val_df)} | test {len(test_df)}  (no lesion shared across splits)")
dist = pd.DataFrame({n: d["dx"].value_counts() for n, d in
                     [("train", train_df), ("val", val_df), ("test", test_df)]}).reindex(CLASS_NAMES)
dist

## 4. Pre-resize images once (much faster training)

Images are resized to a 256-pixel short side and cached on the Colab disk, so each epoch doesn't re-decode full-size files. Takes a couple of minutes, once per session.

In [ ]:
def cache_resized(src, dst, short=256):
    if not os.path.exists(dst):
        img = Image.open(src).convert("RGB")
        w, h = img.size
        s = short / min(w, h)
        if s < 1:
            img = img.resize((round(w * s), round(h * s)), Image.BILINEAR)
        img.save(dst, "JPEG", quality=95)
    return dst

HAM_CACHE = f"{DATA_DIR}/ham_256"
os.makedirs(HAM_CACHE, exist_ok=True)
with ThreadPoolExecutor(8) as ex:
    for split_df in (train_df, val_df, test_df):
        split_df["image_path"] = list(ex.map(
            lambda r: cache_resized(r.src_path, f"{HAM_CACHE}/{r.image_id}.jpg"),
            split_df.itertuples()))
print("HAM10000 cached:", len(os.listdir(HAM_CACHE)), "images")

## 5. PAD-UFES-20: external smartphone test set

Never used for training or model selection. Its diagnoses are mapped onto the same three tiers. ACK, BCC, SCC and BOD are mapped to "moderate" to match HAM10000's `akiec`/`bcc`; state this mapping in the paper as a design choice.

In [ ]:
PAD_TO_TIER = {"NEV": "low", "SEK": "low",
               "ACK": "moderate", "BCC": "moderate", "SCC": "moderate", "BOD": "moderate",
               "MEL": "high"}

pad = pd.read_csv(PAD_CSV)

label_col = next(c for c in pad.columns if c.lower() == "diagnostic")
img_col   = next((c for c in pad.columns if c.lower() in ("img_id", "image_id", "img", "image")), None)
assert img_col is not None, f"Image-id column not found. Columns: {list(pad.columns)}"

pad[label_col] = pad[label_col].astype(str).str.upper().str.strip()
pad = pad[pad[label_col].isin(PAD_TO_TIER)].copy()
pad["tier_int"] = pad[label_col].map(PAD_TO_TIER).map(TIER_TO_INT)

pad["src_path"] = pad[img_col].astype(str).map(lambda s: PAD_IMAGES.get(os.path.splitext(s)[0]))
pad_df = pad.dropna(subset=["src_path"]).reset_index(drop=True)

PAD_CACHE = f"{DATA_DIR}/pad_256"
os.makedirs(PAD_CACHE, exist_ok=True)
with ThreadPoolExecutor(8) as ex:
    pad_df["image_path"] = list(ex.map(
        lambda r: cache_resized(r.src_path, f"{PAD_CACHE}/{os.path.splitext(os.path.basename(r.src_path))[0]}.jpg"),
        pad_df.itertuples()))

print(f"PAD-UFES-20 usable images: {len(pad_df)}")
print(pad_df["tier_int"].map(dict(enumerate(TIER_NAMES))).value_counts())

## 6. Metadata, augmentations, and data loaders

- **Metadata** (age, sex, site) is encoded with a "present" flag. During training it is hidden 30% of the time, so the model also works when metadata is missing (as on PAD-UFES-20, where it's evaluated image-only).
- **Smartphone simulation** degrades an image the way a phone camera would: lower resolution, blur, JPEG artifacts, perspective, and colour/white-balance shifts.
- The sampler balances classes; the loss is **not** additionally class-weighted, to avoid the double-correction bug found earlier.

In [ ]:
SEX_CATS = sorted(train_df["sex"].fillna("unknown").astype(str).unique())
LOC_CATS = sorted(train_df["localization"].fillna("unknown").astype(str).unique())
AGE_MEAN, AGE_STD = train_df["age"].mean(), train_df["age"].std()
META_DIM = 1 + len(SEX_CATS) + len(LOC_CATS) + 1   # age + sex one-hot + site one-hot + present flag

def encode_meta(row, drop=False):
    v = np.zeros(META_DIM, dtype=np.float32)
    if drop:
        return v                                     # all zeros = "metadata missing"
    age = row.get("age", np.nan)
    v[0] = ((age if pd.notna(age) else AGE_MEAN) - AGE_MEAN) / AGE_STD
    sex = row.get("sex", np.nan)
    sex = str(sex) if pd.notna(sex) else "unknown"
    if sex in SEX_CATS:
        v[1 + SEX_CATS.index(sex)] = 1
    loc = row.get("localization", np.nan)
    loc = str(loc) if pd.notna(loc) else "unknown"
    if loc in LOC_CATS:
        v[1 + len(SEX_CATS) + LOC_CATS.index(loc)] = 1
    v[-1] = 1.0
    return v

IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.02),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

class PhoneSim:
    def __call__(self, img):
        w, h = img.size
        if random.random() < 0.8:                    # resolution loss
            f = random.uniform(0.25, 0.6)
            img = img.resize((max(32, int(w * f)), max(32, int(h * f))), Image.BILINEAR).resize((w, h), Image.BILINEAR)
        if random.random() < 0.5:                    # defocus
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 2.0)))
        if random.random() < 0.7:                    # JPEG compression
            buf = io.BytesIO()
            img.save(buf, format="JPEG", quality=random.randint(30, 70))
            buf.seek(0)
            img = Image.open(buf).convert("RGB")
        return img

def white_balance_shift(t):
    return (t * torch.empty(3, 1, 1).uniform_(0.85, 1.15)).clamp(0, 1)

phone_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomPerspective(0.2, p=0.5),
    PhoneSim(),
    transforms.ColorJitter(0.4, 0.4, 0.3, 0.05),
    transforms.ToTensor(), transforms.Lambda(white_balance_shift),
    transforms.Normalize(MEAN, STD),
])

class HamDataset(Dataset):
    def __init__(self, df, train=False, phone=False, meta_drop_p=0.3):
        self.df, self.train, self.phone, self.meta_drop_p = df.reset_index(drop=True), train, phone, meta_drop_p
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["image_path"]).convert("RGB")
        y, t = CLASS_TO_IDX[row["dx"]], int(row["tier_int"])
        if self.train:
            meta = torch.from_numpy(encode_meta(row, drop=random.random() < self.meta_drop_p))
            x_phone = phone_tf(img) if self.phone else torch.zeros(1)
            return train_tf(img), x_phone, meta, y, t
        return eval_tf(img), torch.from_numpy(encode_meta(row)), y, t

class PadDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["image_path"]).convert("RGB")
        return eval_tf(img), torch.zeros(META_DIM), -1, int(row["tier_int"])   # image-only, no 7-class label

BATCH_SIZE = 32
LOADER_KW = dict(num_workers=2, pin_memory=True)

def make_train_loader(phone):
    ds = HamDataset(train_df, train=True, phone=phone)
    counts = train_df["dx"].value_counts()
    weights = train_df["dx"].map(lambda c: 1.0 / counts[c]).values
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, **LOADER_KW)

val_loader  = DataLoader(HamDataset(val_df),  batch_size=64, shuffle=False, **LOADER_KW)
test_loader = DataLoader(HamDataset(test_df), batch_size=64, shuffle=False, **LOADER_KW)
pad_loader  = DataLoader(PadDataset(pad_df),  batch_size=64, shuffle=False, **LOADER_KW)
print(f"META_DIM={META_DIM} | val {len(val_df)} | test {len(test_df)} | PAD {len(pad_df)}")

In [ ]:
# Quick visual check: a training image next to its smartphone-simulated version
def denorm(t):
    return (t * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)).clamp(0, 1)

demo = HamDataset(train_df.sample(4, random_state=1), train=True, phone=True)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    xc, xp, _, y, _ = demo[i]
    axes[0, i].imshow(denorm(xc).permute(1, 2, 0)); axes[0, i].set_title(f"{CLASS_NAMES[y]} (dermoscopic)", fontsize=9)
    axes[1, i].imshow(denorm(xp).permute(1, 2, 0)); axes[1, i].set_title("phone-simulated", fontsize=9)
    axes[0, i].axis("off"); axes[1, i].axis("off")
plt.tight_layout(); plt.show()

## 7. DoRT-Net: model, losses, training and evaluation

In [ ]:
class FiLM(nn.Module):
    """Metadata produces a per-feature scale and shift. Initialised as identity."""
    def __init__(self, meta_dim, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(meta_dim, 64), nn.ReLU(), nn.Linear(64, 2 * dim))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
    def forward(self, z, m):
        gamma, beta = self.net(m).chunk(2, dim=1)
        return z * (1 + gamma) + beta

class PrototypeAttention(nn.Module):
    """Each image attends to learned 'typical case' vectors (per_class per class)."""
    def __init__(self, dim, num_classes, per_class=4, tau=0.1):
        super().__init__()
        self.protos = nn.Parameter(torch.randn(num_classes * per_class, dim) * 0.02)
        self.tau = tau
        self.norm = nn.LayerNorm(dim)
    def forward(self, z):
        sim = F.normalize(z, dim=1) @ F.normalize(self.protos, dim=1).T
        attn = F.softmax(sim / self.tau, dim=1)
        return self.norm(z + attn @ self.protos), attn

class DoRTNet(nn.Module):
    def __init__(self, meta_dim, num_classes=7, emb=256, use_film=True, use_proto=True, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.efficientnet_b0(weights=weights)
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, emb), nn.SiLU())
        self.use_film, self.use_proto = use_film, use_proto
        self.film  = FiLM(meta_dim, emb) if use_film else None
        self.proto = PrototypeAttention(emb, num_classes) if use_proto else None
        self.head  = nn.Sequential(nn.Dropout(0.2), nn.Linear(emb, num_classes))
    def forward(self, x, m):
        z = self.proj(self.pool(self.features(x)).flatten(1))
        if self.use_film:
            z = self.film(z, m)
        attn = None
        if self.use_proto:
            z, attn = self.proto(z)
        return self.head(z), attn

# ---- tier structure and losses ----
TIER_MAP = torch.zeros(len(CLASS_NAMES), 3)            # 7 classes -> 3 tiers
for i, c in enumerate(CLASS_NAMES):
    TIER_MAP[i, TIER_TO_INT[CLASS_TO_TIER[c]]] = 1.0

# rows = true tier, cols = predicted tier. Under-triage (predicting a LOWER tier) costs more.
COST = torch.tensor([[0., 1., 1.],
                     [2., 0., 1.],
                     [4., 2., 0.]])
TIER_MAP_D, COST_D = TIER_MAP.to(DEVICE), COST.to(DEVICE)

def focal_loss(logits, y, gamma=1.5):
    ce = F.cross_entropy(logits.float(), y, reduction="none")
    return (((1 - torch.exp(-ce)) ** gamma) * ce).mean()

def triage_cost_loss(logits, t):
    tier_p = F.softmax(logits.float(), dim=1) @ TIER_MAP_D
    return (tier_p * COST_D[t]).sum(dim=1).mean()          # expected triage cost

def consistency_loss(logits_phone, logits_clean):
    return F.kl_div(F.log_softmax(logits_phone.float(), dim=1),
                    F.softmax(logits_clean.float().detach(), dim=1), reduction="batchmean")

def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

In [ ]:
def set_backbone_trainable(model, from_block):
    for i, block in enumerate(model.features):
        for p in block.parameters():
            p.requires_grad = i >= from_block

def train_epoch(model, loader, opt, scaler, cfg, max_batches=None):
    model.train()
    total, n = 0.0, 0
    for b, (xc, xp, m, y, t) in enumerate(loader):
        if max_batches and b >= max_batches:
            break
        xc, m, y, t = xc.to(DEVICE, non_blocking=True), m.to(DEVICE), y.to(DEVICE), t.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits_c, _ = model(xc, m)
            loss = focal_loss(logits_c, y)
            if cfg["cost"]:
                loss = loss + cfg["lam_cost"] * triage_cost_loss(logits_c, t)
            if cfg["phonesim"]:
                logits_p, _ = model(xp.to(DEVICE, non_blocking=True), m)
                loss = loss + focal_loss(logits_p, y) + cfg["lam_cons"] * consistency_loss(logits_p, logits_c)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        total += loss.item() * y.size(0)
        n += y.size(0)
    return total / max(n, 1)

@torch.no_grad()
def evaluate(model, loader, zero_meta=False):
    model.eval()
    probs, ys, ts = [], [], []
    for x, m, y, t in loader:
        if zero_meta:
            m = torch.zeros_like(m)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits, _ = model(x.to(DEVICE, non_blocking=True), m.to(DEVICE))
        probs.append(F.softmax(logits.float(), dim=1).cpu())
        ys.append(torch.as_tensor(y)); ts.append(torch.as_tensor(t))
    P = torch.cat(probs); Y = torch.cat(ys).numpy(); T = torch.cat(ts).numpy()
    tier_pred = (P @ TIER_MAP).numpy().argmax(1)
    out = {
        "tier_f1": f1_score(T, tier_pred, average="macro", zero_division=0),
        "high_recall": recall_score(T, tier_pred, labels=[2], average=None, zero_division=0)[0],
        "tier_true": T, "tier_pred": tier_pred,
    }
    if (Y >= 0).all():
        out["f1_7"] = f1_score(Y, P.argmax(1).numpy(), average="macro", zero_division=0)
    return out

In [ ]:
EPOCHS_HEAD = 4     # backbone frozen, train new layers
EPOCHS_FT   = 16    # unfreeze EfficientNet blocks 5-8 and fine-tune

def run_experiment(cfg, quick=False):
    name = cfg["name"]
    res_path = f"{RESULTS_DIR}/{name}.json"
    if not quick and os.path.exists(res_path):
        print(f"[{name}] already finished -- loading saved results.")
        with open(res_path) as f:
            return json.load(f)

    seed_everything()
    train_loader = make_train_loader(phone=cfg["phonesim"])
    model = DoRTNet(META_DIM, len(CLASS_NAMES), use_film=cfg["film"], use_proto=cfg["proto"]).to(DEVICE)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    ckpt = "/content/smoke_test.pt" if quick else f"{CKPT_DIR}/{name}.pt"
    phases = [(99, 1e-3, 1 if quick else EPOCHS_HEAD),     # 99 = every backbone block frozen
              (5,  1e-4, 1 if quick else EPOCHS_FT)]
    max_batches = 20 if quick else None

    best, t_start = -1.0, time.time()
    for phase_no, (from_block, lr, epochs) in enumerate(phases, 1):
        set_backbone_trainable(model, from_block)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.01)
        for ep in range(1, epochs + 1):
            t0 = time.time()
            loss = train_epoch(model, train_loader, opt, scaler, cfg, max_batches)
            sched.step()
            v = evaluate(model, val_loader)
            saved = ""
            if v["tier_f1"] > best:                       # model selection on the triage objective
                best = v["tier_f1"]
                torch.save(model.state_dict(), ckpt)
                saved = "  <- saved"
            print(f"[{name}] phase {phase_no} epoch {ep}/{epochs} | loss {loss:.4f} | "
                  f"val tier-F1 {v['tier_f1']:.4f} | high recall {v['high_recall']:.3f} | "
                  f"7-class F1 {v['f1_7']:.4f} | {time.time() - t0:.0f}s{saved}")

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    ham_res   = evaluate(model, test_loader)
    ham_nomet = evaluate(model, test_loader, zero_meta=True)
    pad_res   = evaluate(model, pad_loader, zero_meta=True)

    result = {
        "name": name,
        "ham_tier_f1": round(float(ham_res["tier_f1"]), 4),
        "ham_high_recall": round(float(ham_res["high_recall"]), 4),
        "ham_f1_7class": round(float(ham_res["f1_7"]), 4),
        "ham_tier_f1_no_meta": round(float(ham_nomet["tier_f1"]), 4),
        "pad_tier_f1": round(float(pad_res["tier_f1"]), 4),
        "pad_high_recall": round(float(pad_res["high_recall"]), 4),
        "params_M": round(count_params(model), 2),
        "train_minutes": round((time.time() - t_start) / 60, 1),
    }
    print(f"\n[{name}] TEST RESULTS")
    for k, v in result.items():
        print(f"  {k:22s} {v}")
    print("\nHAM10000 test (tier level):")
    print(classification_report(ham_res["tier_true"], ham_res["tier_pred"], target_names=TIER_NAMES, zero_division=0))
    print("PAD-UFES-20 external (tier level):")
    print(classification_report(pad_res["tier_true"], pad_res["tier_pred"], target_names=TIER_NAMES, zero_division=0))

    if not quick:
        with open(res_path, "w") as f:
            json.dump(result, f, indent=2)
        np.savez(f"{RESULTS_DIR}/{name}_preds.npz",
                 ham_true=ham_res["tier_true"], ham_pred=ham_res["tier_pred"],
                 pad_true=pad_res["tier_true"], pad_pred=pad_res["tier_pred"])
    return result

BASE = dict(lam_cost=0.5, lam_cons=1.0)
CONFIGS = {
    "baseline":    dict(name="baseline",    film=False, proto=False, cost=False, phonesim=False, **BASE),
    "full":        dict(name="full",        film=True,  proto=True,  cost=True,  phonesim=True,  **BASE),
    "no_phonesim": dict(name="no_phonesim", film=True,  proto=True,  cost=True,  phonesim=False, **BASE),
    "no_cost":     dict(name="no_cost",     film=True,  proto=True,  cost=False, phonesim=True,  **BASE),
    "no_proto":    dict(name="no_proto",    film=True,  proto=False, cost=True,  phonesim=True,  **BASE),
    "no_film":     dict(name="no_film",     film=False, proto=True,  cost=True,  phonesim=True,  **BASE),
}
print("Experiments defined:", list(CONFIGS))

## 8. Smoke test (run this first)

Trains for a few batches only. The scores will be meaningless; this just proves every part runs end to end on your GPU. Nothing is saved to Drive.

In [ ]:
_ = run_experiment(CONFIGS["full"], quick=True)

## 9. Experiments

Each run takes roughly an hour on a T4 (the phone-simulation runs process two views per image, so they are the slowest). Run them one at a time; finished runs are saved to Drive and skipped if you re-run the cell.

Priority order if time is short: **full → baseline → no_phonesim**. Those three alone support the paper's main claim.

In [ ]:
results_full = run_experiment(CONFIGS["full"])

In [ ]:
results_baseline = run_experiment(CONFIGS["baseline"])

In [ ]:
results_no_phonesim = run_experiment(CONFIGS["no_phonesim"])

In [ ]:
# Remaining ablations -- one at a time if a session is short
for key in ["no_cost", "no_proto", "no_film"]:
    run_experiment(CONFIGS[key])

## 10. Results table for the paper

In [ ]:
ORDER = ["baseline", "full", "no_phonesim", "no_cost", "no_proto", "no_film"]
LABELS = {"baseline": "EfficientNet-B0 baseline", "full": "DoRT-Net (full)",
          "no_phonesim": "- phone-sim consistency", "no_cost": "- under-triage cost loss",
          "no_proto": "- prototype attention", "no_film": "- FiLM metadata"}

rows = []
for key in ORDER:
    path = f"{RESULTS_DIR}/{key}.json"
    if os.path.exists(path):
        with open(path) as f:
            r = json.load(f)
        r["model"] = LABELS[key]
        rows.append(r)

if rows:
    table = pd.DataFrame(rows)[["model", "ham_tier_f1", "ham_high_recall", "ham_tier_f1_no_meta",
                                "pad_tier_f1", "pad_high_recall", "ham_f1_7class", "params_M", "train_minutes"]]
    table.columns = ["Model", "HAM tier F1", "HAM high recall", "HAM tier F1 (no meta)",
                     "PAD tier F1", "PAD high recall", "HAM 7-class F1", "Params (M)", "Train (min)"]
    table.to_csv(f"{RESULTS_DIR}/results_table.csv", index=False)
    display(table)
    print("Saved:", f"{RESULTS_DIR}/results_table.csv")
else:
    print("No finished experiments yet.")

In [ ]:
# Confusion matrices: baseline vs DoRT-Net, in-domain and external
available = [k for k in ["baseline", "full"] if os.path.exists(f"{RESULTS_DIR}/{k}_preds.npz")]
if available:
    fig, axes = plt.subplots(len(available), 2, figsize=(10, 4.5 * len(available)), squeeze=False)
    for r, key in enumerate(available):
        d = np.load(f"{RESULTS_DIR}/{key}_preds.npz")
        for c, (split, title) in enumerate([("ham", "HAM10000 test"), ("pad", "PAD-UFES-20 (smartphone)")]):
            cm = confusion_matrix(d[f"{split}_true"], d[f"{split}_pred"], labels=[0, 1, 2])
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[r, c],
                        xticklabels=TIER_NAMES, yticklabels=TIER_NAMES, cbar=False)
            axes[r, c].set_title(f"{LABELS[key]} -- {title}", fontsize=10)
            axes[r, c].set_xlabel("Predicted tier"); axes[r, c].set_ylabel("True tier")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/confusion_matrices.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Run the baseline and full experiments first.")

## 11. Model size and inference speed

This answers the "5 models are too slow" concern with measured numbers. Two of DoRT-Net's four parts (under-triage cost loss and phone-simulation consistency) are **used only during training**, so they add nothing at inference. FiLM and prototype attention add about 0.04M parameters to the EfficientNet-B0 baseline.

The earlier soft-vote ensemble needed a CNN, a Radius-GIN, two LightGBM models and a Balanced Random Forest for every prediction. The GIN also needed a similarity graph built against every stored training image. DoRT-Net needs one image, one forward pass and one checkpoint file.

This cell runs even before training finishes: speed doesn't depend on the learned weights, so it uses the trained checkpoint if it exists and random weights otherwise.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

def load_for_benchmark(key):
    cfg = CONFIGS[key]
    m = DoRTNet(META_DIM, len(CLASS_NAMES), use_film=cfg["film"], use_proto=cfg["proto"], pretrained=False)
    ckpt = f"{CKPT_DIR}/{key}.pt"
    if os.path.exists(ckpt):
        m.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=True))
    return m.eval().to(DEVICE)

@torch.no_grad()
def gmacs(model):
    model = model.cpu()
    with FlopCounterMode(display=False) as fc:
        model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE), torch.zeros(1, META_DIM))
    return fc.get_total_flops() / 2e9

@torch.no_grad()
def latency_ms(model, device, runs=50):
    model = model.to(device)
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    m = torch.zeros(1, META_DIM, device=device)
    for _ in range(10):
        model(x, m)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(runs):
        model(x, m)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / runs * 1000

rows = []
for key in ["baseline", "full"]:
    mdl = load_for_benchmark(key)
    row = {
        "Model": LABELS[key],
        "Params (M)": round(count_params(mdl), 2),
        "GMACs / image": round(gmacs(mdl), 2),
        "Size (MB)": round(sum(p.numel() * p.element_size() for p in mdl.parameters()) / 1e6, 1),
        "CPU ms / image": round(latency_ms(mdl, torch.device("cpu"), runs=20), 1),
    }
    if torch.cuda.is_available():
        row["GPU ms / image"] = round(latency_ms(mdl, torch.device("cuda")), 2)
    rows.append(row)

efficiency = pd.DataFrame(rows)
efficiency.to_csv(f"{RESULTS_DIR}/efficiency.csv", index=False)
display(efficiency)
print("Saved:", f"{RESULTS_DIR}/efficiency.csv")

## 12. Triage advisor demo

The trained model plus a small rules layer: low confidence or a red-flag symptom raises the tier by one step, and every decision comes with its reasons.

In [ ]:
RED_FLAGS = {"bleeding", "rapid_growth", "irregular_border", "color_change", "persistent_itching", "diameter_over_6mm"}
CARE = {
    "low":      "Low concern. Monitor for changes; no urgent action needed.",
    "moderate": "Moderate concern. Book a dermatologist visit within 2-4 weeks.",
    "high":     "High concern. Seek dermatology care as soon as possible.",
}

@torch.no_grad()
def triage(model, image_path, meta_row=None, symptoms=None, conf_threshold=0.5):
    x = eval_tf(Image.open(image_path).convert("RGB")).unsqueeze(0).to(DEVICE)
    meta = encode_meta(meta_row) if meta_row is not None else np.zeros(META_DIM, dtype=np.float32)
    logits, _ = model(x, torch.from_numpy(meta).unsqueeze(0).to(DEVICE))
    p = F.softmax(logits.float(), dim=1).cpu()[0]
    tier_p = (p @ TIER_MAP).numpy()
    tier = int(tier_p.argmax()); conf = float(tier_p[tier])
    reasons = [f"Model tier: {TIER_NAMES[tier]} (confidence {conf:.2f}); "
               f"most similar class: {CLASS_NAMES[int(p.argmax())]}."]
    if conf < conf_threshold and tier < 2:
        tier += 1
        reasons.append(f"Confidence below {conf_threshold}: escalated one tier as a precaution.")
    flags = sorted(s for s, v in (symptoms or {}).items() if v and s in RED_FLAGS)
    if flags and tier < 2:
        tier += 1
        reasons.append(f"Red-flag symptoms reported ({', '.join(flags)}): escalated one tier.")
    return {"tier": TIER_NAMES[tier], "advice": CARE[TIER_NAMES[tier]], "reasons": reasons,
            "tier_probabilities": dict(zip(TIER_NAMES, np.round(tier_p, 3).tolist())),
            "disclaimer": "Preliminary triage suggestion only -- not a medical diagnosis."}

if os.path.exists(f"{CKPT_DIR}/full.pt"):
    advisor = load_for_benchmark("full")
    row = test_df.iloc[0]
    print("HAM10000 example -- true tier:", TIER_NAMES[row["tier_int"]])
    print(json.dumps(triage(advisor, row["image_path"], meta_row=row), indent=2))
    prow = pad_df.iloc[0]
    print("\nSmartphone example -- true tier:", TIER_NAMES[prow["tier_int"]])
    print(json.dumps(triage(advisor, prow["image_path"], symptoms={"bleeding": True}), indent=2))
else:
    print("Train the full model first (Section 9).")

## What goes in the paper

- **Proposed method:** DoRT-Net (Section 7), one EfficientNet-B0-based network with four components.
- **Main table:** Section 10's `results_table.csv`. The key comparisons are baseline vs full on PAD-UFES-20, and full vs "- phone-sim consistency" to show that component is what improves smartphone robustness.
- **Figures:** `confusion_matrices.png` (300 dpi) from your Drive folder.
- **Efficiency:** parameters and ms/image from Section 11, answering the "too many models" concern directly.
- **Method notes to state:** lesion-level split (no lesion shared across splits), model selection on validation tier-F1, PAD-UFES-20 used only for testing and evaluated without metadata, and the PAD diagnosis-to-tier mapping.

Numbers from the old notebook used a different (image-level) split and are not directly comparable to these.

In [ ]:
# ================= Reviewer-response analysis (no training; reloads saved checkpoints) =================
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, average_precision_score, precision_recall_curve
from scipy.stats import binomtest

C = COST.numpy()
TIER_MAP_CPU = TIER_MAP.cpu()

def load_ckpt(key):
    cfg = CONFIGS[key]
    m = DoRTNet(META_DIM, len(CLASS_NAMES), use_film=cfg["film"], use_proto=cfg["proto"], pretrained=False).to(DEVICE)
    m.load_state_dict(torch.load(f"{CKPT_DIR}/{key}.pt", map_location=DEVICE, weights_only=True))
    return m.eval()

@torch.no_grad()
def tier_probs(model, loader):
    qs, ts = [], []
    for x, m, y, t in loader:
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits, _ = model(x.to(DEVICE, non_blocking=True), m.to(DEVICE))
        qs.append(F.softmax(logits.float(), dim=1).cpu() @ TIER_MAP_CPU)
        ts.append(torch.as_tensor(t))
    return torch.cat(qs).numpy(), torch.cat(ts).numpy()

def metrics(t, pred):
    return {
        "tier_f1": f1_score(t, pred, average="macro", zero_division=0),
        "high_recall": recall_score(t, pred, labels=[2], average=None, zero_division=0)[0],
        "high_precision": precision_score(t, pred, labels=[2], average=None, zero_division=0)[0],
        "mean_cost": C[t, pred].mean(),
        "under_%": 100 * (pred < t).mean(),
        "over_%": 100 * (pred > t).mean(),
        "missed_high": int(((t == 2) & (pred < 2)).sum()),
    }

def thresh_pred(q, tau):
    pred = q[:, :2].argmax(1)
    pred[q[:, 2] >= tau] = 2
    return pred

bayes_pred = lambda q: (q @ C).argmin(1)

def ece(q, t, bins=15):
    conf, acc = q.max(1), (q.argmax(1) == t)
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            e += sel.mean() * abs(acc[sel].mean() - conf[sel].mean())
    return e

def escalate(q, flags=None, thr=0.5):
    pred, conf = q.argmax(1).copy(), q.max(1)
    by_conf = (conf < thr) & (pred < 2)
    pred[by_conf] += 1
    by_flag = np.zeros(len(pred), bool)
    if flags is not None:
        by_flag = flags & (pred < 2)
        pred[by_flag] += 1
    return pred, by_conf.mean() * 100, by_flag.mean() * 100

# ---------- 1. tier probabilities for val / test / PAD ----------
PROBS = {}
for key in ["baseline", "full"]:
    mdl = load_ckpt(key)
    PROBS[key] = {s: tier_probs(mdl, ld) for s, ld in [("val", val_loader), ("ham", test_loader), ("pad", pad_loader)]}
    saved = np.load(f"{RESULTS_DIR}/{key}_preds.npz")
    for s in ("ham", "pad"):
        agree = (PROBS[key][s][0].argmax(1) == saved[f"{s}_pred"]).mean()
        print(f"sanity [{key}] {s}: reloaded predictions match saved ones for {100 * agree:.2f}% of images")
    np.savez(f"{RESULTS_DIR}/{key}_tier_probs.npz", **{f"{s}_q": PROBS[key][s][0] for s in PROBS[key]},
             **{f"{s}_t": PROBS[key][s][1] for s in PROBS[key]})
    del mdl

results, rows = {}, []
def add(name, split, t, pred):
    r = {"decision": name, "split": split, **{k: round(float(v), 4) for k, v in metrics(t, pred).items()}}
    rows.append(r)
    return r

# ---------- 2. threshold-free ranking quality of the high tier (W1) ----------
print("\n=== High-tier ranking quality (threshold-free) ===")
for key in ["baseline", "full"]:
    for s in ("ham", "pad"):
        q, t = PROBS[key][s]
        auc, ap = roc_auc_score(t == 2, q[:, 2]), average_precision_score(t == 2, q[:, 2])
        results[f"{key}_{s}_high_auc"], results[f"{key}_{s}_high_ap"] = auc, ap
        print(f"{key:8s} {s.upper()}: ROC-AUC={auc:.4f}  PR-AUC={ap:.4f}")

# ---------- 3. decision rules compared at fair operating points (W1) ----------
grid = np.linspace(0.01, 0.99, 99)
qv_b, tv = PROBS["baseline"]["val"]
qv_f, _ = PROBS["full"]["val"]
full_val_recall = metrics(tv, qv_f.argmax(1))["high_recall"]
tau_match = max([tau for tau in grid if metrics(tv, thresh_pred(qv_b, tau))["high_recall"] >= full_val_recall], default=0.01)
tau_cost = {k: min(grid, key=lambda tau: metrics(tv, thresh_pred(PROBS[k]["val"][0], tau))["mean_cost"]) for k in ["baseline", "full"]}
results.update(tau_match_baseline=tau_match, tau_cost_baseline=tau_cost["baseline"], tau_cost_full=tau_cost["full"])

for s in ("ham", "pad"):
    qb, t = PROBS["baseline"][s]
    qf, _ = PROBS["full"][s]
    add("baseline argmax", s, t, qb.argmax(1))
    add("full argmax", s, t, qf.argmax(1))
    add(f"baseline, threshold matched to full recall on val (tau={tau_match:.2f})", s, t, thresh_pred(qb, tau_match))
    add(f"baseline, cost-optimal threshold on val (tau={tau_cost['baseline']:.2f})", s, t, thresh_pred(qb, tau_cost["baseline"]))
    add(f"full, cost-optimal threshold on val (tau={tau_cost['full']:.2f})", s, t, thresh_pred(qf, tau_cost["full"]))
    add("baseline, Bayes cost rule", s, t, bayes_pred(qb))
    add("full, Bayes cost rule", s, t, bayes_pred(qf))

# ---------- 4. paired statistics: full vs baseline, argmax (W3) ----------
print("\n=== Paired tests, full vs baseline (argmax decisions) ===")
rng = np.random.default_rng(0)
for s in ("ham", "pad"):
    qb, t = PROBS["baseline"][s]
    qf, _ = PROBS["full"][s]
    pb, pf = qb.argmax(1), qf.argmax(1)
    hb, hf = pb[t == 2] == 2, pf[t == 2] == 2
    only_f, only_b = int((hf & ~hb).sum()), int((hb & ~hf).sum())
    p_mc = binomtest(only_f, only_f + only_b, 0.5).pvalue if only_f + only_b else 1.0
    boot = {"mean_cost": [], "tier_f1": [], "high_recall": []}
    for _ in range(2000):
        i = rng.integers(0, len(t), len(t))
        mf, mb = metrics(t[i], pf[i]), metrics(t[i], pb[i])
        for k in boot:
            boot[k].append(mf[k] - mb[k])
    ci = {k: np.percentile(v, [2.5, 97.5]) for k, v in boot.items()}
    results[f"{s}_mcnemar"] = {"only_full": only_f, "only_baseline": only_b, "p": p_mc}
    results[f"{s}_bootstrap_ci_full_minus_baseline"] = {k: [float(a), float(b)] for k, (a, b) in ci.items()}
    print(f"{s.upper()}: melanomas caught only by full={only_f}, only by baseline={only_b}, exact McNemar p={p_mc:.4g}")
    for k, (a, b) in ci.items():
        print(f"      95% bootstrap CI of (full - baseline) {k:12s}: [{a:+.4f}, {b:+.4f}]")

# ---------- 5. calibration ----------
print("\n=== Calibration (tier-level ECE, 15 bins) ===")
for key in ["baseline", "full"]:
    for s in ("ham", "pad"):
        results[f"{key}_{s}_ece"] = ece(*PROBS[key][s])
        print(f"{key:8s} {s.upper()}: ECE={results[f'{key}_{s}_ece']:.4f}")

# ---------- 6. escalation layer (W8): HAM = confidence rule only; PAD = confidence + real patient symptoms ----------
def yes(col):
    return pad_df[col].astype(str).str.strip().str.upper().eq("TRUE").values if col in pad_df else np.zeros(len(pad_df), bool)
diam = np.fmax(pd.to_numeric(pad_df.get("diameter_1"), errors="coerce").values,
               pd.to_numeric(pad_df.get("diameter_2"), errors="coerce").values)
FLAG = {"bleeding": yes("bleed"), "rapid_growth": yes("grew"), "color_change": yes("changed"),
        "itching_persistent": yes("itch"), "diameter_over_6mm": np.nan_to_num(diam, nan=0) > 6}
t_pad = PROBS["full"]["pad"][1]
print("\n=== PAD-UFES-20 red-flag prevalence by true tier (% of lesions) ===")
print(pd.DataFrame({f: [100 * v[t_pad == k].mean() for k in range(3)] for f, v in FLAG.items()}, index=TIER_NAMES).round(1))
flags_all = np.any(np.stack(list(FLAG.values())), axis=0)
flags_no_itch = np.any(np.stack([v for f, v in FLAG.items() if f != "itching_persistent"]), axis=0)

esc_rows = []
for key in ["baseline", "full"]:
    for s, variants in [("ham", [("model only", None, False), ("+ confidence rule", None, True)]),
                        ("pad", [("model only", None, False), ("+ confidence rule", None, True),
                                 ("+ confidence + all red flags", flags_all, True),
                                 ("+ confidence + red flags except itch", flags_no_itch, True)])]:
        q, t = PROBS[key][s]
        for name, flags, use_conf in variants:
            if use_conf:
                pred, pct_conf, pct_flag = escalate(q, flags)
            else:
                pred, pct_conf, pct_flag = q.argmax(1), 0.0, 0.0
            esc_rows.append({"model": key, "split": s, "decision": name, "escalated_by_conf_%": round(pct_conf, 1),
                             "escalated_by_flags_%": round(pct_flag, 1),
                             **{k: round(float(v), 4) for k, v in metrics(t, pred).items()}})

# ---------- 7. PR curves figure ----------
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (s, title) in zip(axes, [("ham", "HAM10000 test"), ("pad", "PAD-UFES-20 (smartphone)")]):
    for key, label in [("baseline", "EfficientNet-B0 baseline"), ("full", "DoRT-Net")]:
        q, t = PROBS[key][s]
        prec, rec, _ = precision_recall_curve(t == 2, q[:, 2])
        ax.plot(rec, prec, label=f"{label} (AP={results[f'{key}_{s}_high_ap']:.3f})")
        op = metrics(t, q.argmax(1))
        ax.scatter([op["high_recall"]], [op["high_precision"]], s=25)
    ax.set_title(f"High tier - {title}", fontsize=10); ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/pr_curves_high_tier.png", dpi=300, bbox_inches="tight"); plt.show()

# ---------- save + print ----------
decision_table = pd.DataFrame(rows)
escalation_table = pd.DataFrame(esc_rows)
decision_table.to_csv(f"{RESULTS_DIR}/review_decision_rules.csv", index=False)
escalation_table.to_csv(f"{RESULTS_DIR}/review_escalation.csv", index=False)
with open(f"{RESULTS_DIR}/review_analysis.json", "w") as f:
    json.dump(results, f, indent=2, default=float)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 70)
print("\n=== Decision rules at fair operating points ==="); print(decision_table.to_string(index=False))
print("\n=== Escalation layer ==="); print(escalation_table.to_string(index=False))
print("\nSaved to", RESULTS_DIR, ": review_decision_rules.csv, review_escalation.csv, review_analysis.json, pr_curves_high_tier.png")